# SEED-VII EEG-Conformer 端到端流水线

**新增第 0 步：先把 32 个分卷在 ModelScope 上 stream-merge 成一个完整 zip 并 push 回去；之后所有逻辑都基于这个完整 zip。**

数据源三选一（按推荐顺序）：
- **C. ModelScope 单一合并 zip**（推荐）— 已合并完后用 HTTP Range 流式读取，**磁盘 0 字节**
- **B. ModelScope 多分卷** — 兼容性 fallback；不需要预合并
- **A. 本地分卷**

脚本：
0. `scripts/merge_and_upload.py` — **在 ModelScope 上 stream-merge** 32 分卷为 1 个 zip 并 push 回（任意时刻 ≤ 1 卷 + 16 MB 内存）
1. `scripts/preprocess_to_npz.py` — 流式预处理 → npz（含 trial-level 划分）
2. `scripts/train.py` — 训练 / 续训（断点续训 + 10h 软超时）
3. `scripts/encode.py` — 编码导出 embedding
4. `scripts/upload_to_modelscope.py` — 上传其它产物

辅助：`scripts/ms_fetch.py` — `list / fetch-info / fetch-one / fetch-volumes`。

> 严格遵循 `Design.md` 的所有原则。

## 0. 环境与路径

In [ ]:
import os, sys, subprocess, json, pathlib
REPO = pathlib.Path.cwd().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
print('REPO =', REPO)

# ====== 数据源选择 ======
USE_MERGED_ZIP     = True            # 推荐：合并后用 Range 流式读
USE_MULTI_VOLUMES  = False           # 不合并，直接读 32 分卷
USE_LOCAL_VOLUMES  = False           # 本地分卷目录

MS_DATASET         = 'DEREKVERSE/SEED-VII'
MS_REVISION        = 'master'
MS_TOKEN           = os.environ.get('MODELSCOPE_API_TOKEN', '')
VOL_PATTERN        = '*.zip.*'
MERGED_ZIP_PATH    = 'SEED-VII.zip'              # 合并后在数据集中的路径
SAVE_INFO_GLOB     = 'save_info/*_save_info.csv'

# Stage 0 暂存目录（合并时只需要 ~1 卷 ≈ 5.4GB）
MERGE_SCRATCH      = '/workspace/_merge_scratch'

# 本地分卷模式才需要
VOLUMES_DIR        = '/data/seed_vii_volumes'
SAVE_INFO_LOCAL    = '/data/seed_vii_save_info'

# 工作区（100 GB 持久化盘）
WORK         = pathlib.Path('/workspace/seed_vii'); WORK.mkdir(parents=True, exist_ok=True)
MS_SCRATCH   = str(WORK / '_ms_volumes_cache')
NPZ_OUT      = str(WORK / 'preprocessed' / 'seed_vii.npz')
RUN_DIR      = str(WORK / 'runs_seed_vii')
ENC_OUT      = str(WORK / 'encoded' / 'embeddings.npz')

def run(cmd, env=None):
    print('$', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=str(REPO), env={**os.environ, **(env or {})})
    if res.returncode != 0:
        raise SystemExit(f'cmd failed with code {res.returncode}')
PY = sys.executable
print('python =', PY)

## 0. Stream-merge 32 分卷 → 1 个 zip 并 push 回 ModelScope（**新**）

**完整 zip 永远不落盘**——分卷一个一个拉到 `MERGE_SCRATCH`，喂给 SHA-256 / HTTP PUT，立即 unlink。

- Stage 1: 流式 SHA-256（下载 ≈160GB，最多保留 1 卷）
- Stage 2: 询问 ModelScope LFS 该 blob 是否已存在（若重跑则会跳过 Stage 3，**天然续传**）
- Stage 3: 流式上传（再下载 ≈160GB，再上传 ≈160GB；最多保留 1 卷）
- Stage 4: 创建 commit 把 LFS blob 关联到 `SEED-VII.zip`
- Stage 5: 验证远端 listing

状态写入 `MERGE_SCRATCH/_merge_upload_state.json`，**断点续传**。

In [ ]:
if USE_MERGED_ZIP:
    if not MS_TOKEN:
        raise SystemExit('Please set os.environ["MODELSCOPE_API_TOKEN"] before merging+uploading.')
    run([PY, 'scripts/merge_and_upload.py',
         '--dataset', MS_DATASET,
         '--pattern', VOL_PATTERN,
         '--path-in-repo', MERGED_ZIP_PATH,
         '--scratch-dir', MERGE_SCRATCH,
         '--revision', MS_REVISION,
         '--commit-message', 'Add merged SEED-VII.zip via stream-merge-upload'])
else:
    print('Skipping Stage 0 (merge+upload). Will use existing source.')

## 1. 流式预处理 → 单个 npz

**先切分（trial-level），再做窗口/标准化** → 杜绝数据泄漏。
每 clip 居中 60% + 4 秒窗口 + 50% 重叠 + 按通道 z-score。

三种来源的磁盘占用：
- C. 合并 zip + Range：**0 字节** + 256 MB 内存 LRU 缓存
- B. 多分卷 LazyConcatStream：最多 2 卷 ≈ 11 GB（默认 `--ms-max-resident-volumes 2`）
- A. 本地：0 额外

In [ ]:
common = ['--output', NPZ_OUT,
          '--val-ratio', '0.1', '--test-ratio', '0.1',
          '--split-unit', 'trial',
          '--max-windows-per-trial', '60']

if USE_MERGED_ZIP:
    run([PY, 'scripts/preprocess_to_npz.py',
         '--ms-single-zip', MS_DATASET,
         '--ms-single-zip-path', MERGED_ZIP_PATH,
         '--ms-revision', MS_REVISION,
         '--ms-save-info-include', SAVE_INFO_GLOB,
         '--ms-scratch-dir', MS_SCRATCH,
         '--ms-range-cache-mb', '256',
         '--ms-range-chunk-mb', '8',
         *common])
elif USE_MULTI_VOLUMES:
    run([PY, 'scripts/preprocess_to_npz.py',
         '--ms-dataset', MS_DATASET, '--ms-revision', MS_REVISION,
         '--pattern', VOL_PATTERN,
         '--ms-scratch-dir', MS_SCRATCH,
         '--ms-max-resident-volumes', '2',
         '--ms-save-info-include', SAVE_INFO_GLOB,
         *common])
elif USE_LOCAL_VOLUMES:
    run([PY, 'scripts/preprocess_to_npz.py',
         '--volumes-dir', VOLUMES_DIR,
         '--pattern', VOL_PATTERN,
         '--save-info-dir', SAVE_INFO_LOCAL,
         *common])

## 2. 训练（先分类预训练，再联合训练；10h 自动安全保存）

In [ ]:
run([PY, 'scripts/train.py',
     '--data', NPZ_OUT,
     '--output-dir', RUN_DIR,
     '--device', 'auto', '--amp',
     '--pretrain-epochs', '10',
     '--max-epochs', '200',
     '--batch-size', '64',
     '--lr', '2e-4', '--min-lr', '1e-5',
     '--sample-weight-mode', 'continuous',
     '--max-runtime-hours', '10'])

In [ ]:
# 续训
# run([PY, 'scripts/train.py', '--data', NPZ_OUT, '--output-dir', RUN_DIR,
#      '--device', 'auto', '--amp', '--resume', '--max-runtime-hours', '10'])

## 3. 编码导出

In [ ]:
run([PY, 'scripts/encode.py',
     '--data', NPZ_OUT,
     '--checkpoint', str(pathlib.Path(RUN_DIR) / 'best_encoder.pt'),
     '--output', ENC_OUT,
     '--feature-type', 'projected',
     '--device', 'auto', '--amp'])
import numpy as np; z = np.load(ENC_OUT, allow_pickle=True)
print({k: getattr(z[k], 'shape', z[k]) for k in z.files})